# 06 - Cleaning Eurostat Status Changes for Cameroonian Citizens

This notebook cleans Eurostat data on immigration status changes for Cameroonian citizens.

Analytical role:

- supports Q3 on post-arrival trajectories
- captures administrative transitions after arrival
- keeps status change indicators separate from permits, residence and citizenship indicators
- prepares a standardized file for the final analytical tables

The `period` field is treated as an operational grouping for comparison across available reference years, not as a causal Covid-19 measure.


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(columns=[2, 4, 6, 8, 10], errors="ignore")
sheet1_raw = sheet1_raw.drop(index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 44, 45, 46, 47, 48], errors="ignore").reset_index(drop=True)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)
sheet1_raw = sheet1_raw.drop(index=[1, 2])

sheet1_raw

In [ ]:
# Use the first row as the header
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Rename TIME to destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Remove the GEO (Labels) row if it is still present
sheet1_raw = sheet1_raw[
    sheet1_raw["destination_country"] != "GEO (Labels)"
]

# Reshape from wide format to long format
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="permit"
)

# Clean data types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["permit"] = (
    pd.to_numeric(sheet1_raw["permit"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Reorder columns
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "permit"]
]

sheet1_raw


In [ ]:
# Rename permit to permits
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

In [ ]:
# Add source and nature columns
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "status_change"

# Add the operational reference period
def classify_covid_period(year):
    if year < 2020:
        return "pre_covid"
    elif year in [2020, 2021]:
        return "covid"
    else:
        return "post_covid"

sheet1_raw["period_covid"] = sheet1_raw["year"].apply(classify_covid_period)

# Reorder columns
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "permits",
        "source",
        "nature",
        "period_covid"
    ]
]

sheet1_raw


In [ ]:
sheet1_raw = sheet1_raw.rename(
    columns={"period_covid": "covid_period"}
)

sheet1_raw

In [ ]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_status_changes_cleaned.csv",
    index=False
)